# 02 · Nhận dạng thực thể (NER)

Gán nhãn cho **từng từ**:

| Từ | Nhãn |
|---|---|
| Lan | B-PER |
| sống | O |
| ở | O |
| Hà | B-LOC |
| Nội | I-LOC |

`B-` = bắt đầu thực thể, `I-` = phần tiếp theo, `O` = không phải thực thể.

Lý thuyết: xem `knowledge/02_ner.md`

In [ ]:
# !pip install -U transformers datasets accelerate seqeval

## 0. Import

In [ ]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from seqeval.metrics import f1_score, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 1. Cấu hình

In [ ]:
TRAIN_FILE = None    # file CoNLL: mỗi dòng "từ nhãn", câu cách nhau bằng dòng trống
VALID_FILE = None
TEST_FILE = None

MODEL_NAME = "xlm-roberta-base"   # cần tokenizer "fast" -> dùng XLM-R, không dùng PhoBERT ở notebook này
MAX_LEN = 256

## 2. Đọc dữ liệu
Mỗi câu = danh sách từ + danh sách nhãn.

In [ ]:
def read_conll(path):
    sentences = []
    words = []
    tags = []

    for line in open(path, encoding="utf-8"):
        line = line.strip()

        if line == "":                       # dòng trống = hết câu
            if words:
                sentences.append({"words": words, "tags": tags})
            words = []
            tags = []
            continue

        parts = line.split()
        words.append(parts[0])               # cột đầu là từ
        tags.append(parts[-1])               # cột cuối là nhãn

    if words:
        sentences.append({"words": words, "tags": tags})
    return pd.DataFrame(sentences)

In [ ]:
def load_demo_data():
    data = load_dataset("eriktks/conll2003")
    tag_names = data["train"].features["ner_tags"].feature.names

    def convert(split):
        rows = []
        for example in split:
            tags = [tag_names[t] for t in example["ner_tags"]]
            rows.append({"words": example["tokens"], "tags": tags})
        return pd.DataFrame(rows)

    return convert(data["train"]), convert(data["validation"]), convert(data["test"])


if TRAIN_FILE is None:
    train_df, valid_df, test_df = load_demo_data()
else:
    train_df = read_conll(TRAIN_FILE)
    valid_df = read_conll(VALID_FILE)
    test_df = read_conll(TEST_FILE)

print(len(train_df), len(valid_df), len(test_df))
print(list(zip(train_df["words"][0], train_df["tags"][0])))

In [ ]:
all_tags = set()
for tags in train_df["tags"]:
    all_tags.update(tags)

tag_list = sorted(all_tags)
tag2id = {tag: i for i, tag in enumerate(tag_list)}
id2tag = {i: tag for tag, i in tag2id.items()}
print(tag_list)

## 3. Căn nhãn theo sub-token

Tokenizer có thể cắt 1 từ thành nhiều mảnh:

```
từ     :  Washington
token  :  ▁Washing   ton
nhãn   :  B-LOC      -100     <- chỉ mảnh đầu giữ nhãn, mảnh sau = -100 (bỏ qua khi tính loss)
```
`word_ids()` cho biết mỗi token thuộc từ thứ mấy.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align(batch):
    encoded = tokenizer(batch["words"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)

    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = encoded.word_ids(batch_index=i)
        labels = []
        previous_word = None

        for word_id in word_ids:
            if word_id is None:                  # token đặc biệt <s>, </s>
                labels.append(-100)
            elif word_id != previous_word:       # mảnh đầu tiên của từ
                labels.append(tag2id[tags[word_id]])
            else:                                # mảnh tiếp theo của cùng từ
                labels.append(-100)
            previous_word = word_id

        all_labels.append(labels)

    encoded["labels"] = all_labels
    return encoded

In [ ]:
train_ds = Dataset.from_pandas(train_df).map(tokenize_and_align, batched=True, remove_columns=["words", "tags"])
valid_ds = Dataset.from_pandas(valid_df).map(tokenize_and_align, batched=True, remove_columns=["words", "tags"])
test_ds = Dataset.from_pandas(test_df).map(tokenize_and_align, batched=True, remove_columns=["words", "tags"])

## 4. Chấm điểm bằng seqeval
Tính F1 theo **thực thể** (đúng cả ranh giới lẫn loại), không phải theo từng token.

In [ ]:
def logits_to_tags(logits, label_ids):
    predictions = logits.argmax(axis=-1)
    true_tags = []
    pred_tags = []

    for pred_row, label_row in zip(predictions, label_ids):
        sentence_true = []
        sentence_pred = []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                sentence_true.append(id2tag[l])
                sentence_pred.append(id2tag[p])
        true_tags.append(sentence_true)
        pred_tags.append(sentence_pred)

    return true_tags, pred_tags


def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    true_tags, pred_tags = logits_to_tags(logits, label_ids)
    return {"f1": f1_score(true_tags, pred_tags)}

## 5. Train

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(tag_list), id2label=id2tag, label2id=tag2id
)

args = TrainingArguments(
    output_dir="ner_output",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 6. Dự đoán test + điểm chi tiết từng loại thực thể
Nếu file test của đề **không có nhãn**: khi đọc, đặt tạm mọi nhãn = `"O"`. Khi đó điểm in ra không có ý nghĩa, chỉ dùng `pred_tags`.

In [ ]:
output = trainer.predict(test_ds)
true_tags, pred_tags = logits_to_tags(output.predictions, output.label_ids)

print(classification_report(true_tags, pred_tags, digits=4))

## 7. Sửa chuỗi nhãn không hợp lệ
`I-LOC` không thể đứng ngay sau `O` → đổi thành `B-LOC`.

In [ ]:
def fix_bio(tags):
    fixed = []
    for tag in tags:
        if tag.startswith("I-"):
            previous = fixed[-1] if fixed else "O"
            if previous == "O" or previous[2:] != tag[2:]:
                tag = "B-" + tag[2:]
        fixed.append(tag)
    return fixed


pred_tags_fixed = [fix_bio(tags) for tags in pred_tags]
print("F1 trước:", f1_score(true_tags, pred_tags))
print("F1 sau  :", f1_score(true_tags, pred_tags_fixed))

## 8. Lưu bài nộp
Mỗi dòng `từ nhãn`, câu cách nhau bằng dòng trống.

In [ ]:
with open("submission.txt", "w", encoding="utf-8") as f:
    for words, tags in zip(test_df["words"], pred_tags_fixed):
        for word, tag in zip(words, tags):
            f.write(f"{word} {tag}\n")
        f.write("\n")

⚠️ Nếu câu dài hơn `MAX_LEN` thì phần cuối bị cắt và số nhãn dự đoán sẽ ít hơn số từ → tăng `MAX_LEN` hoặc tách câu.

---
## 9. Nâng cao
- `xlm-roberta-large` thường hơn base 1–3 điểm F1.
- Sau khi biết số epoch tốt nhất: gộp train + valid rồi train lại.
- Train 3 seed khác nhau, cộng logits rồi mới argmax.